# 🛍️ Customer Segmentation & Personalized Marketing
### Data Science / AI Solution — SME Retail Proposal

**Business Goal:** Maximize revenue by targeting the right customer with the right promotion using RFM + K-Means Clustering.

| Section | Content |
|---|---|
| 0 | Setup & Imports |
| 1 | Mock Data Generation |
| 2 | Data Dictionary |
| 3 | EDA & Data Quality Check |
| 4 | RFM Feature Engineering |
| 5 | K-Means Clustering |
| 6 | Segment Profiling & Naming |
| 7 | Personalized Promotion Mapping |
| 8 | Mock Output & Dashboard |
| 9 | Workflow Diagram |
| 10 | Pseudo-code Summary |
| 11 | AI Tools Used |

> 🤖 **AI Tools Used:** Claude (Anthropic) — code scaffolding, business logic review. All outputs manually verified.


## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"

print("✅ Libraries loaded successfully")

## 1. Mock Data Generation

**Why mock data?**
- Demonstrate end-to-end pipeline without real customer PII
- Validate feature engineering and clustering logic
- Provide reproducible starting point for the real project

**Assumptions:**
- 500 customers, 12 months of transactions, 4 product categories
- ~3–25 transactions per customer depending on segment type
- ~30% of transactions include a promotion


In [ ]:
N_CUSTOMERS = 500
N_PRODUCTS  = 60
N_STORES    = 5
N_PROMOS    = 8
START_DATE  = datetime(2024, 1, 1)
END_DATE    = datetime(2024, 12, 31)

# ── Customer Master ──────────────────────────────────────────────────────────
segments_true = np.random.choice(
    ["High-Value", "Mid-Regular", "Occasional", "Churned", "New"],
    p=[0.10, 0.25, 0.30, 0.20, 0.15],
    size=N_CUSTOMERS,
)
customer_master = pd.DataFrame({
    "customer_id":         [f"C{str(i).zfill(4)}" for i in range(1, N_CUSTOMERS + 1)],
    "customer_taxonomies": np.random.choice(["Premium", "Regular", "Budget", "Corporate"], size=N_CUSTOMERS),
    "_true_segment":       segments_true,
})

# ── Product Master ───────────────────────────────────────────────────────────
categories = ["Electronics", "Clothing", "Food & Beverage", "Home & Living"]
product_master = pd.DataFrame({
    "product_id":         [f"P{str(i).zfill(3)}" for i in range(1, N_PRODUCTS + 1)],
    "price":              np.round(np.random.uniform(50, 3000, N_PRODUCTS), 2),
    "product_taxonomies": np.random.choice(categories, size=N_PRODUCTS),
})

# ── Store Master ─────────────────────────────────────────────────────────────
store_master = pd.DataFrame({
    "store_id":         [f"S{i}" for i in range(1, N_STORES + 1)],
    "store_taxonomies": ["Bangkok CBD", "Bangkok Suburb", "Chiang Mai", "Phuket", "Online"],
})

# ── Promotion Master ─────────────────────────────────────────────────────────
promo_master = pd.DataFrame({
    "promotion_id": [f"PROMO{i:02d}" for i in range(1, N_PROMOS + 1)],
    "discount":     np.round(np.random.choice([0.05, 0.10, 0.15, 0.20, 0.30], size=N_PROMOS), 2),
    "product_id":   np.random.choice(product_master["product_id"], size=N_PROMOS),
    "start_date":   pd.date_range(START_DATE, periods=N_PROMOS, freq="45D"),
    "end_date":     pd.date_range(START_DATE + timedelta(days=20), periods=N_PROMOS, freq="45D"),
})

# ── Sales Transactions ───────────────────────────────────────────────────────
freq_map = {
    "High-Value":  (15, 25), "Mid-Regular": (8, 15),
    "Occasional":  (3, 8),   "Churned":     (1, 3), "New": (1, 4),
}
rows = []
for _, cust in customer_master.iterrows():
    lo, hi = freq_map[cust["_true_segment"]]
    for _ in range(np.random.randint(lo, hi + 1)):
        txn_date = START_DATE + timedelta(days=int(np.random.uniform(0, (END_DATE - START_DATE).days)))
        product  = product_master.sample(1).iloc[0]
        qty      = np.random.randint(1, 5)
        promo    = promo_master.sample(1).iloc[0] if np.random.rand() < 0.3 else None
        rows.append({
            "datetime":     txn_date,
            "product_id":   product["product_id"],
            "price":        product["price"],
            "qty":          qty,
            "customer_id":  cust["customer_id"],
            "promotion_id": promo["promotion_id"] if promo is not None else None,
            "store_id":     np.random.choice(store_master["store_id"]),
            "po_id":        f"PO{np.random.randint(10000, 99999)}",
        })

sales = pd.DataFrame(rows)
sales["revenue"] = sales["price"] * sales["qty"]

# Introduce ~5% missing customer_id (guest transactions)
mask = np.random.rand(len(sales)) < 0.05
sales.loc[mask, "customer_id"] = np.nan

print(f"✅ Customers:    {len(customer_master):,}")
print(f"✅ Products:     {len(product_master):,}")
print(f"✅ Transactions: {len(sales):,}")
print(f"✅ Missing customer_id: {sales['customer_id'].isna().sum()} rows (~5% guest)")

## 2. Data Dictionary

In [ ]:
data_dict = {
    "Table": [
        "sales","sales","sales","sales","sales","sales","sales","sales",
        "customer_master","customer_master",
        "product_master","product_master","product_master",
        "promo_master","promo_master","promo_master","promo_master","promo_master",
    ],
    "Column": [
        "datetime","product_id","price","qty","customer_id","promotion_id","store_id","po_id",
        "customer_id","customer_taxonomies",
        "product_id","price","product_taxonomies",
        "promotion_id","discount","product_id","start_date","end_date",
    ],
    "Type": [
        "datetime","str","float","int","str","str","str","str",
        "str","str","str","float","str","str","float","str","date","date",
    ],
    "Description": [
        "Transaction timestamp",
        "Product identifier (FK → product_master)",
        "Unit selling price (THB)",
        "Quantity sold",
        "Customer identifier (FK → customer_master); ~5% null = guest",
        "Promotion applied (nullable)",
        "Store identifier (FK → store_master)",
        "Purchase order reference",
        "Customer identifier (PK)",
        "Customer segment label (Premium/Regular/Budget/Corporate)",
        "Product identifier (PK)",
        "Standard product price (THB)",
        "Product category (Electronics/Clothing/Food & Beverage/Home & Living)",
        "Promotion identifier (PK)",
        "Discount rate (0.0–1.0)",
        "Eligible product (FK → product_master)",
        "Promotion start date",
        "Promotion end date",
    ],
    "Used_For_RFM": [
        "✓ Recency/Frequency","✓ join","✓ Monetary","✓ Monetary","✓ join key",
        "✓ promo analysis","○","○",
        "✓ join key","○ optional",
        "✓ join","✓ Monetary","✓ category pref",
        "○","✓ ROI calc","✓ join","✓ overlap","✓ overlap",
    ],
}
pd.set_option("display.max_colwidth", 70)
pd.DataFrame(data_dict)

## 3. EDA & Data Quality Check

Checking:
- Missing values
- Revenue distribution by category
- Monthly revenue trend
- Transaction frequency per customer


In [ ]:
print("Shape:", sales.shape)
print("Date range:", sales["datetime"].min().date(), "→", sales["datetime"].max().date())
print(f"Total revenue: {sales['revenue'].sum():,.0f} THB")
print("\nMissing values:")
print(sales.isnull().sum()[sales.isnull().sum() > 0])

In [ ]:
sales_with_cat = sales.merge(product_master[["product_id","product_taxonomies"]], on="product_id", how="left")
rev_by_cat = sales_with_cat.groupby("product_taxonomies")["revenue"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("EDA — Sales Overview", fontsize=14, fontweight="bold")

rev_by_cat.plot(kind="bar", ax=axes[0], color=["#0EA5E9","#0D9488","#10B981","#F59E0B"])
axes[0].set_title("Revenue by Category")
axes[0].set_xlabel("")
axes[0].set_ylabel("Revenue (THB)")
axes[0].tick_params(axis="x", rotation=30)

sales["month"] = sales["datetime"].dt.to_period("M")
sales.groupby("month")["revenue"].sum().plot(ax=axes[1], color="#0EA5E9", marker="o", linewidth=2)
axes[1].set_title("Monthly Revenue Trend")
axes[1].tick_params(axis="x", rotation=45)

txn_per_cust = sales.dropna(subset=["customer_id"]).groupby("customer_id").size()
axes[2].hist(txn_per_cust, bins=20, color="#0EA5E9", edgecolor="white")
axes[2].set_title("Transactions per Customer")
axes[2].set_xlabel("# Transactions")

plt.tight_layout()
plt.show()

## 4. RFM Feature Engineering

| Feature | Definition | Business Meaning |
|---|---|---|
| **Recency** | Days since last purchase | Lower = more engaged |
| **Frequency** | # unique orders (po_id) | Higher = more loyal |
| **Monetary** | Total spend (THB) | Higher = more valuable |

**Data cleaning steps:**
1. Drop guest transactions (null customer_id)
2. Deduplicate on (datetime, product_id, customer_id)


In [ ]:
SNAPSHOT_DATE = END_DATE + timedelta(days=1)

sales_clean = sales.dropna(subset=["customer_id"]).copy()
sales_clean = sales_clean.drop_duplicates(subset=["datetime","product_id","customer_id"])

rfm = (
    sales_clean.groupby("customer_id")
    .agg(
        last_purchase=("datetime", "max"),
        frequency=("po_id", "nunique"),
        monetary=("revenue", "sum"),
    )
    .reset_index()
)
rfm["recency"] = (SNAPSHOT_DATE - rfm["last_purchase"]).dt.days
rfm = rfm[["customer_id","recency","frequency","monetary"]]

# Quintile-based RFM scores (1–5)
rfm["R_score"] = pd.qcut(rfm["recency"], q=5, labels=[5,4,3,2,1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1,2,3,4,5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"].rank(method="first"), q=5, labels=[1,2,3,4,5]).astype(int)
rfm["RFM_total"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

print(f"✅ RFM table: {len(rfm)} customers")
rfm.describe().round(2)

## 5. K-Means Clustering

**Steps:**
1. Scale features with `StandardScaler`
2. Choose k using Elbow method + Silhouette Score
3. Fit final K-Means model


In [ ]:
features = ["recency","frequency","monetary"]
scaler = StandardScaler()
X = scaler.fit_transform(rfm[features])

inertias, sil_scores = [], []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Choosing Optimal k", fontsize=13, fontweight="bold")

axes[0].plot(list(k_range), inertias, "o-", color="#0EA5E9", linewidth=2)
axes[0].axvline(x=5, color="#EF4444", linestyle="--", alpha=0.7, label="k=5 chosen")
axes[0].set_title("Elbow Method (Inertia)")
axes[0].set_xlabel("k")
axes[0].legend()

axes[1].plot(list(k_range), sil_scores, "o-", color="#10B981", linewidth=2)
axes[1].axvline(x=5, color="#EF4444", linestyle="--", alpha=0.7, label="k=5 chosen")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_k = 5
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm["cluster"] = km_final.fit_predict(X)
best_sil = silhouette_score(X, rfm["cluster"])

print(f"✅ k = {best_k}")
print(f"✅ Silhouette Score = {best_sil:.4f}  (target > 0.45 on real data)")
print("\nCluster distribution:")
print(rfm["cluster"].value_counts().sort_index())

## 6. Segment Profiling & Naming

Auto-name segments based on RFM profile logic:
- **Champions**: High Monetary + High Frequency
- **Loyal Customers**: Low Recency + High Frequency  
- **At Risk / Churned**: Very High Recency
- **New / Occasional**: Low Frequency
- **Potential Growth**: Mid everything


In [ ]:
profile = rfm.groupby("cluster")[["recency","frequency","monetary"]].mean().round(1)
profile["size"] = rfm.groupby("cluster").size()
profile["pct"]  = (profile["size"] / len(rfm) * 100).round(1)

def name_segment(row):
    if row["monetary"] > profile["monetary"].quantile(0.75) and row["frequency"] > profile["frequency"].median():
        return "🌟 Champions"
    elif row["recency"] < profile["recency"].quantile(0.33) and row["frequency"] > profile["frequency"].median():
        return "💎 Loyal Customers"
    elif row["recency"] > profile["recency"].quantile(0.75):
        return "😴 At Risk / Churned"
    elif row["frequency"] < profile["frequency"].quantile(0.33):
        return "🌱 New / Occasional"
    else:
        return "📈 Potential Growth"

profile["segment_name"] = profile.apply(name_segment, axis=1)
rfm = rfm.merge(profile[["segment_name"]], left_on="cluster", right_index=True)

profile[["segment_name","recency","frequency","monetary","size","pct"]]

In [ ]:
palette = ["#0EA5E9","#10B981","#F59E0B","#EF4444","#8B5CF6"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Customer Segments — RFM Clusters", fontsize=13, fontweight="bold")

axes[0].scatter(rfm["recency"], rfm["monetary"], c=rfm["cluster"], cmap="tab10", alpha=0.6, s=30)
axes[0].set_xlabel("Recency (days)")
axes[0].set_ylabel("Monetary (THB)")
axes[0].set_title("Recency vs Monetary")
handles = [mpatches.Patch(color=plt.cm.tab10(i/10), label=profile.loc[i,"segment_name"]) for i in range(best_k)]
axes[0].legend(handles=handles, fontsize=8)

axes[1].scatter(rfm["frequency"], rfm["monetary"], c=rfm["cluster"], cmap="tab10", alpha=0.6, s=30)
axes[1].set_xlabel("Frequency (# orders)")
axes[1].set_ylabel("Monetary (THB)")
axes[1].set_title("Frequency vs Monetary")

plt.tight_layout()
plt.show()

## 7. Personalized Promotion Mapping

Rule-based starting point → future work: ML-based uplift modeling


In [ ]:
PROMO_MAP = {
    "🌟 Champions":         {"action": "VIP Early Access",    "discount": "5–10%",  "channel": "Push Notification", "rationale": "Already loyal — preserve margin"},
    "💎 Loyal Customers":   {"action": "Cross-sell Bundle",   "discount": "10–15%", "channel": "Email",             "rationale": "Ready to try new categories"},
    "📈 Potential Growth":  {"action": "Engagement Promo",    "discount": "15%",    "channel": "Line OA",           "rationale": "Nudge frequency with targeted offer"},
    "🌱 New / Occasional":  {"action": "Welcome Series",      "discount": "20%",    "channel": "SMS",               "rationale": "Build habit early"},
    "😴 At Risk / Churned": {"action": "Win-back Campaign",   "discount": "30%+",   "channel": "Email + SMS",       "rationale": "High discount justified by churn cost avoidance"},
}

promo_df = pd.DataFrame(PROMO_MAP).T.reset_index().rename(columns={"index": "segment_name"})
rfm_output = rfm.merge(promo_df, on="segment_name", how="left")
promo_df

## 8. Mock Output & Dashboard

In [ ]:
summary = (
    rfm_output.groupby("segment_name")
    .agg(customers=("customer_id","count"), avg_revenue=("monetary","mean"),
         avg_frequency=("frequency","mean"), action=("action","first"), channel=("channel","first"))
    .round(1).reset_index()
)
summary["est_revenue_uplift"] = (summary["avg_revenue"] * summary["customers"] * 0.15).round(0)
print(f"💰 Estimated Total Revenue Uplift (15% scenario): {summary['est_revenue_uplift'].sum():,.0f} THB\n")
summary[["segment_name","customers","avg_revenue","avg_frequency","action","est_revenue_uplift"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Mock Dashboard — Segment Overview", fontsize=13, fontweight="bold")
colors = ["#0EA5E9","#10B981","#F59E0B","#EF4444","#8B5CF6"]

axes[0].barh(summary["segment_name"], summary["customers"], color=colors[:len(summary)], edgecolor="white")
axes[0].set_title("Customers per Segment")
axes[0].set_xlabel("# Customers")

axes[1].barh(summary["segment_name"], summary["avg_revenue"], color=colors[:len(summary)], edgecolor="white")
axes[1].set_title("Avg Revenue per Segment (THB)")
axes[1].set_xlabel("Avg Revenue (THB)")

plt.tight_layout()
plt.show()

In [ ]:
# Save output CSV
output_cols = ["customer_id","recency","frequency","monetary",
               "R_score","F_score","M_score","RFM_total",
               "cluster","segment_name","action","discount","channel"]
rfm_output[output_cols].to_csv("rfm_output.csv", index=False)
print("✅ rfm_output.csv saved")
rfm_output[output_cols].head(10)

## 9. Workflow Diagram

```
┌─────────────────┐
│  Business Goal  │  Increase Revenue per Customer + Promo ROI
└────────┬────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│ DATA COLLECTION                                             │
│  Sales Transaction + Customer + Product + Promotion tables  │
└────────┬────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│ DATA QUALITY & EDA                                          │
│  • Null check (customer_id ~5%)   • Duplicate removal       │
│  • Revenue distribution           • Monthly trend           │
└────────┬────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│ FEATURE ENGINEERING — RFM                                   │
│  Recency = days since last purchase                         │
│  Frequency = # unique orders                                │
│  Monetary = total spend (THB)                               │
└────────┬────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│ MODELING — K-Means Clustering (k=5)                         │
│  StandardScaler → KMeans → Elbow + Silhouette validation    │
└────────┬────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│ SEGMENT PROFILING & NAMING                                  │
│  🌟 Champions │ 💎 Loyal │ 📈 Growth │ 🌱 New │ 😴 At Risk  │
└────────┬────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│ PERSONALIZED OFFER ENGINE                                   │
│  Right discount + Right channel + Right timing per segment  │
└────────┬────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│ MEASURE & ITERATE                                           │
│  KPIs: Revenue Per Customer │ Promo ROI │ Retention Rate    │
│  A/B Test: Targeted promo vs Broadcast                      │
└─────────────────────────────────────────────────────────────┘
```


## 10. Pseudo-code Summary

```python
def run_customer_segmentation(sales_df, snapshot_date):
    # Step 1: Clean
    clean = drop_guest_transactions(sales_df)
    clean = remove_duplicates(clean)

    # Step 2: RFM
    rfm = compute_rfm(clean, snapshot_date)
    # recency   = (snapshot_date - last_purchase).days
    # frequency = nunique(po_id)
    # monetary  = sum(price * qty)

    # Step 3: Scale
    X = StandardScaler().fit_transform(rfm[['recency','frequency','monetary']])

    # Step 4: Cluster
    k = choose_k_via_elbow_and_silhouette(X, k_range=range(2, 9))
    rfm['cluster'] = KMeans(n_clusters=k).fit_predict(X)

    # Step 5: Profile & name
    profile = rfm.groupby('cluster').mean()
    rfm['segment_name'] = profile.apply(name_segment, axis=1)

    # Step 6: Map promotions
    rfm = rfm.merge(PROMO_MAP, on='segment_name')

    return rfm  # → feed to dashboard / CRM / email platform
```


## 11. AI Tools Used & Verification

| Tool | How Used | How Verified |
|---|---|---|
| Claude (Anthropic) — claude-sonnet-4-6 | Code scaffolding, business logic review, structure design, segment naming logic, promotion mapping rules | All code traced line-by-line; distributions validated against SME benchmarks; charts visually inspected |

**Limitations:**
- Mock data uses simplified distributions; real data may have more skewness, seasonality, and category-specific patterns
- Segment names assigned by simple rule — will be reviewed with business stakeholders before production use
